In [8]:
import os
os.environ['TRANSFORMERS_CACHE'] = './cache/'
import transformers
from torch import nn
import torch
from be_great.multihead_models import MOEModelForCausalLM
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, Trainer, TrainingArguments, EarlyStoppingCallback, BitsAndBytesConfig
from transformers import DataCollatorForTokenClassification
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from torch.nn.utils.rnn import pad_sequence
from matplotlib import pyplot as plt
from tqdm import tqdm 
import argparse
import datetime
import json
from be_great import GReaT
from be_great.great_dataset import GReaTDataset, GReaTDataCollator
from be_great.great_trainer import GReaTTrainer
import re
from shutil import copy
from sklearn import preprocessing, pipeline, ensemble, compose
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from accelerate import PartialState

# modelname = 'meta-llama/Meta-Llama-3-8B'
modelname='distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(modelname, padding_side='left')
special_tokens_dict = {"bos_token": "<BOS>", 'eos_token': '<EOS>'}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
model = transformers.AutoModelForCausalLM.from_pretrained(modelname)
model.resize_token_embeddings(len(tokenizer))
pytorch_total_params = sum(p.numel() for p in model.parameters())

/home/sonia/miniconda3/envs/greatt/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
pytorch_total_params #eoh llama 8B

81914112

In [10]:
head_params = sum(p.numel() for p in model.lm_head.parameters())
head_params

38598912

In [11]:
mh_heads = 6
emh_params = pytorch_total_params + (mh_heads-1)*head_params
emh_params

274908672

In [ ]:
 81914112
274908672